# Case Study — Diffusion content / style

Replicates [`experiments/diffusion_content_style`](../../experiments/diffusion_content_style) as
a package-usage example, driven through the shipped backend client ([`client.py`](client.py)) with
`run_smart_batch` — the same pattern as [`case_study_2.ipynb`](case_study_2.ipynb).

A curated grid of `content x style` prompts through the `daam` attributor on SDXL. Generation
runs **inside** the backend, so every image and its per-token attention heatmaps become an
interactive job: in the frontend, open a job and click the **content token** (e.g. `cow`) vs the
**style token** (e.g. `Rembrandt`) to compare their overlays.

To reproduce the reference figures, each prompt uses a fixed `seed` and the reference generation
settings — SDXL default guidance (`5.0`) and **no** negative prompt (the backend otherwise
defaults to 7.5 + a negative prompt).

**Prerequisites**: the LumiXAI backend running on `http://localhost:8000` **with GPU access**.
SDXL is heavy — this takes a few minutes. No images are written to disk; they live in the backend
Job History.

In [ ]:
from client import Client

client = Client()          # connects to http://localhost:8000
client.clear_history()     # start from a clean Job History

In [ ]:
DIFFUSION_MODEL = "stabilityai/stable-diffusion-xl-base-1.0"
REFERENCE_GUIDANCE_SCALE = 5.0
REFERENCE_NEGATIVE_PROMPT = ""   # "" disables the negative prompt

# (prompt, reference_seed) — one prompt per content/style pair; first is the project mascot.
DIFFUSION_EXAMPLES = [
    ("a cow with Rembrandt style", 12997),
    ("a painting of a person in the Cubism style", 7),
    ("a painting of a zebra in the Ukiyo e style", 1126),
    ("a car in the Pop Art style", 8119),
    ("a Vincent van Gogh painting of a bird", 4749),
    ("a Impressionism painting of a pizza", 6662),
    ("a Salvador Dali painting of a bicycle", 4098),
    ("a painting of a elephant in the Art Nouveau style", 1003),
]

diffusion_batch = [
    {
        "source": "huggingface",
        "model": DIFFUSION_MODEL,
        "attributor": "daam",
        "prompt": prompt,
        "seed": seed,
        "ignore_special_tokens": True,
        "guidance_scale": REFERENCE_GUIDANCE_SCALE,
        "negative_prompt": REFERENCE_NEGATIVE_PROMPT,
    }
    for prompt, seed in DIFFUSION_EXAMPLES
]

print(f"Submitting {len(diffusion_batch)} diffusion jobs "
      f"(guidance={REFERENCE_GUIDANCE_SCALE}, no negative prompt)...")
results_diff = client.run_smart_batch(diffusion_batch, sort_strategy="none")
print("Diffusion batch completed.")

In [ ]:
# Images and heatmaps live in the backend Job History — open the frontend to explore them.
completed = sum(1 for r in results_diff if r and r.get("status") == "completed")
print(f"{completed}/{len(diffusion_batch)} images generated.")
for (prompt, seed), r in zip(DIFFUSION_EXAMPLES, results_diff):
    status = r.get("status") if r else "no result"
    print(f"  [{status}] seed={seed:<6} {prompt}")
print("\nOpen the LumiXAI frontend Job History and click a job -> content token vs style token.")

## Cleanup

Unload the model and free VRAM once you are done.

In [ ]:
client.free_memory()